# Ilera-ATLAS · 03b_build: freeze the benchmark, build the training files

**Runbook Part 6.** CPU only (Accelerator **None**, Internet **On**, secret `HF_TOKEN`). Run it twice:

| Run | `MODE` | When | Reads | Writes (Hugging Face `ilera-data`) |
|---|---|---|---|---|
| 1 | `"bench"` | **Mon 5 Oct**, before the baseline | `reviewed/review_en_final.csv`, `reviewed/review_ha_yo_final.csv` (bench rows) | `bench/ilera_bench.jsonl`, `bench/afrimedqa_ids.txt`, `bench/BENCH_SHA256.txt` |
| 2 | `"train"` | **Tue 6 Oct**, before run A | the same two sheets (now with train rows) + optional `reviewed/referral_cases_final.csv` + the frozen bench | `train.jsonl`, `val.jsonl`, `train_stats.json` |

**The freeze is protected:** run 1 refuses to overwrite an existing benchmark unless you set
`OVERWRITE_BENCH = True` (only do that if you also record the change in `BENCH_CHANGES.md` and re-run
the baseline). Run 2 checks the benchmark hash before using it.

**Leak protection in run 2:** a training row is removed if it *is* a benchmark item, is a *translation*
of a benchmark item, comes from a probe chunk, or its question is a near-duplicate (RapidFuzz
token-set ratio ≥ 85) of any benchmark question.

In [ ]:
# ---- Config -------------------------------------------------------------
HF_USER = "YOUR_HF_USERNAME"          # <- change this
DATA_REPO = f"{HF_USER}/ilera-data"
MODE = "bench"                         # "bench" (Mon 5 Oct) or "train" (Tue 6 Oct)
LANGS = ["ha", "yo"]                   # translated languages in the benchmark
N_BENCH_EN, N_BENCH_LANG, N_PROBE_MIN = 300, 150, 60   # plan targets
AFRIMED_N = 300                        # external MCQ test items
AFRIMED_SPECIALTIES = ["Pediatrics", "Obstetrics_and_Gynecology", "Infectious_Disease",
                       "Internal_Medicine", "Emergency_Medicine"]
AYA_FRACTION = 0.10                    # general-language rows added to training
DEDUP_THRESHOLD = 85                   # RapidFuzz token_set_ratio vs benchmark questions
VAL_FRACTION, VAL_MIN = 0.05, 150
MAX_TOKENS = 2048                      # must match MAX_LEN in the training notebook
FORCE = False                          # True = continue even if counts are below the plan targets
OVERWRITE_BENCH = False                # True = replace an already frozen benchmark (avoid!)
UPLOAD = True
LOCAL_DIR = None                       # offline testing: folder mirroring the HF dataset layout

In [ ]:
import os, subprocess, sys
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)   # deprecated Kaggle default -> removes the FutureWarning
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"         # its replacement: faster model downloads
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "datasets", "huggingface_hub"], capture_output=True, text=True)
if r.returncode != 0:                               # pip prints harmless dependency warnings; show them only on failure
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install failed")
print("packages installed")

In [ ]:
import os
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_USER != "YOUR_HF_USERNAME", "Set HF_USER in the Config cell"
if UPLOAD or not LOCAL_DIR:
    assert HF_TOKEN, "No HF_TOKEN found: add it as a Kaggle/Colab secret or env var"
    os.environ["HF_TOKEN"] = HF_TOKEN

## 1. Helpers

In [ ]:
import csv, hashlib, json, random, re, collections

def fetch(path, required=True):
    """Local path of a file from the dataset repo (or LOCAL_DIR). None if optional and missing."""
    if LOCAL_DIR:
        p = os.path.join(LOCAL_DIR, path)
        if os.path.exists(p):
            return p
    else:
        try:
            from huggingface_hub import hf_hub_download
            return hf_hub_download(DATA_REPO, path, repo_type="dataset", token=HF_TOKEN)
        except Exception as e:
            if required:
                raise FileNotFoundError(f"{path} not found in {DATA_REPO}: {e}")
            return None
    if required:
        raise FileNotFoundError(path)
    return None

def read_csv(path):
    return list(csv.DictReader(open(path, encoding="utf-8-sig")))

def upload(files, msg):
    if not UPLOAD:
        print("(dry run) would upload:", files)
        return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for local, remote in files:
        api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=DATA_REPO,
                        repo_type="dataset", commit_message=msg)
    print("uploaded:", [r for _, r in files])

def sha256(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

ACTION_RE = re.compile(r"^ACTION:\s*(TREAT|REFER|URGENT)\s*$", re.M)

def answer_problems(ans):
    p = []
    if not ACTION_RE.search(ans):
        p.append("no valid ACTION line")
    if "WHAT TO DO:" not in ans:
        p.append("no WHAT TO DO")
    if not re.search(r"^SOURCE:\s*\S", ans, re.M):
        p.append("no SOURCE line")
    return p

## 2. Load the reviewed sheets
Rows with `status` other than `ok` / `fixed` are ignored. Rows whose answer lost the required format
are listed and dropped, so fix them in the sheet if the list is long.

In [ ]:
SYSTEM = open(fetch("config/system_prompt.txt"), encoding="utf-8").read().strip()
probe_chunks = set(open(fetch("raw/probe_chunks.txt"), encoding="utf-8").read().split())

def load_reviewed(path, default_lang=None):
    if path is None:
        return []
    out, bad = [], []
    for r in read_csv(path):
        if r.get("status", "").strip().lower() not in ("ok", "fixed"):
            continue
        r["lang"] = (r.get("lang") or default_lang or "en").strip()
        r["en_id"] = r.get("en_id") or r["id"]
        r["answer"] = r["answer"].replace("\r\n", "\n").strip()
        m = ACTION_RE.search(r["answer"])
        r["action"] = m.group(1) if m else r.get("action", "").strip().upper()
        probs = answer_problems(r["answer"])
        if probs or not r["question"].strip():
            bad.append((r["id"], probs or ["empty question"]))
            continue
        out.append(r)
    if bad:
        print(f"  dropped {len(bad)} rows with a broken answer format, e.g.:", bad[:5])
    return out

en = load_reviewed(fetch("reviewed/review_en_final.csv"), "en")
hy = load_reviewed(fetch("reviewed/review_ha_yo_final.csv", required=False))
ref = load_reviewed(fetch("reviewed/referral_cases_final.csv", required=False), "en")
for name, rows in (("English", en), ("Hausa/Yoruba", hy), ("referral cases", ref)):
    print(f"{name}: {len(rows)} usable rows", dict(collections.Counter((r['lang'], r['split'] if 'split' in r else '-') for r in rows)))
assert en, "no usable English rows"

## 3. Key facts
Each benchmark item lists 2–4 facts a correct answer must contain. A fact that does not appear in the
item's **own gold answer** can never be matched, so it is removed (and listed). Items left with no fact
are excluded from the benchmark.

In [ ]:
def clean_facts(r):
    facts = [k.strip() for k in r.get("key_facts", "").split(";") if k.strip()]
    ok = [k for k in facts if k.lower() in r["answer"].lower()]
    return ok, [k for k in facts if k not in ok]

## 4. MODE = 'bench': build and freeze the benchmark (Mon 5 Oct)

In [ ]:
if MODE == "bench":
    if not OVERWRITE_BENCH and fetch("bench/ilera_bench.jsonl", required=False):
        raise RuntimeError("A frozen benchmark already exists on Hugging Face. Set OVERWRITE_BENCH = True "
                           "only if you really mean to replace it (then log it in BENCH_CHANGES.md "
                           "and re-run the baseline).")
    rng = random.Random(5)
    removed_facts, no_fact = [], []

    def bench_item(r, subset):
        facts, dropped = clean_facts(r)
        if dropped:
            removed_facts.append((r["id"], dropped))
        if not facts:
            no_fact.append(r["id"])
            return None
        return {"id": r["id"], "en_id": r["en_id"], "subset": subset, "lang": r["lang"],
                "question": r["question"].strip(), "gold_action": r["action"], "key_facts": facts,
                "gold_answer": r["answer"], "chunk_id": r["chunk_id"], "source": r.get("source", "")}

    # translated benchmark items (parallel to English)
    lang_items = [bench_item(r, "main") for r in hy if r["split"] == "bench" and r["lang"] in LANGS]
    lang_items = [x for x in lang_items if x]
    parallel_en = {x["en_id"] for x in lang_items}

    # English: every original of a translated item, then random fill up to N_BENCH_EN
    en_pool = {r["id"]: r for r in en if r["split"] == "bench"}
    missing_originals = parallel_en - set(en_pool)
    assert not missing_originals, f"translated items whose English original is not a reviewed bench row: {list(missing_originals)[:5]}"
    en_ids = sorted(parallel_en)
    rest = sorted(set(en_pool) - parallel_en)
    rng.shuffle(rest)
    en_items = []
    for i in en_ids + rest:
        if len(en_items) >= N_BENCH_EN and i not in parallel_en:
            break
        x = bench_item(en_pool[i], "main")
        if x:
            en_items.append(x)
    probe_items = [x for x in (bench_item(r, "probe") for r in en if r["split"] == "probe") if x]
    bench = en_items + lang_items + probe_items

    counts = collections.Counter((x["subset"], x["lang"]) for x in bench)
    print("benchmark:", dict(counts), "| total", len(bench))
    if removed_facts:
        print(f"key facts not found in their own answer were removed from {len(removed_facts)} items, e.g.", removed_facts[:3])
    if no_fact:
        print(f"{len(no_fact)} items excluded because no key fact matched their answer, e.g.", no_fact[:5])
    short = []
    if counts[("main", "en")] < N_BENCH_EN: short.append(f"English {counts[('main', 'en')]}/{N_BENCH_EN}")
    for l in LANGS:
        if counts[("main", l)] < N_BENCH_LANG: short.append(f"{l} {counts[('main', l)]}/{N_BENCH_LANG}")
    if sum(v for (s, _), v in counts.items() if s == "probe") < N_PROBE_MIN: short.append("probe below target")
    if short:
        msg = "Below plan targets: " + ", ".join(short)
        assert FORCE, msg + ". Finish reviews, or set FORCE = True to freeze anyway (and report it)."
        print("WARNING:", msg)
    assert len({x["id"] for x in bench}) == len(bench), "duplicate benchmark ids"
    assert not {x["chunk_id"] for x in en_items + lang_items} & probe_chunks, "a main item comes from a probe chunk"

### External test: AfriMed-QA multiple-choice subset
Public CSV from the AfriMed-QA GitHub repository (CC-BY-SA-4.0). We keep only the **test** split,
multiple-choice questions, primary-care-relevant specialties, Nigerian contributors first. Only the
`sample_id`s are stored (we do not re-publish the questions); the evaluation notebook reloads them.

In [ ]:
AFRIMED_URL = ("https://raw.githubusercontent.com/intron-innovation/AfriMed-QA/master/data/"
               "afri_med_qa_15k_v2.5_phase_2_15275.csv")

def load_afrimed():
    import io, urllib.request
    req = urllib.request.Request(AFRIMED_URL, headers={"User-Agent": "Mozilla/5.0"})
    rows = list(csv.DictReader(io.StringIO(urllib.request.urlopen(req, timeout=300).read().decode("utf-8"))))
    keep = []
    for r in rows:
        if r["question_type"] != "mcq" or r["split"] != "test" or r["specialty"] not in AFRIMED_SPECIALTIES:
            continue
        try:
            opts = {k: v for k, v in json.loads(r["answer_options"]).items() if v and v.strip().upper() != "N/A"}
        except Exception:
            continue
        if len(opts) >= 3 and r["correct_answer"] in opts:
            keep.append(r)
    return keep

if MODE == "bench":
    amq = load_afrimed()
    ng = [r for r in amq if r["country"] == "NG"]
    other = [r for r in amq if r["country"] != "NG"]
    rng.shuffle(ng); rng.shuffle(other)
    amq_pick = (ng + other)[:AFRIMED_N]
    print(f"AfriMed-QA candidates: {len(amq)} ({len(ng)} Nigerian) -> picked {len(amq_pick)}",
          dict(collections.Counter(r["specialty"] for r in amq_pick)))

### Freeze

In [ ]:
if MODE == "bench":
    os.makedirs("bench", exist_ok=True)
    bench.sort(key=lambda x: (x["subset"], x["lang"], x["id"]))
    write_jsonl("bench/ilera_bench.jsonl", bench)
    open("bench/afrimedqa_ids.txt", "w").write("\n".join(r["sample_id"] for r in amq_pick))
    stats = {"items": len(bench), "by_subset_lang": {f"{s}/{l}": n for (s, l), n in sorted(counts.items())},
             "gold_action": dict(collections.Counter(x["gold_action"] for x in bench)),
             "afrimedqa": len(amq_pick), "afrimedqa_source": AFRIMED_URL}
    json.dump(stats, open("bench/bench_stats.json", "w"), indent=2)
    hashes = {f: sha256(f"bench/{f}") for f in ("ilera_bench.jsonl", "afrimedqa_ids.txt")}
    open("bench/BENCH_SHA256.txt", "w").write("".join(f"{h}  {f}\n" for f, h in hashes.items()))
    print(json.dumps(stats, indent=1))
    print(open("bench/BENCH_SHA256.txt").read())
    upload([(f"bench/{f}", f"bench/{f}") for f in
            ("ilera_bench.jsonl", "afrimedqa_ids.txt", "bench_stats.json", "BENCH_SHA256.txt")],
           "03b_build: FREEZE benchmark")
    print("Benchmark frozen. Commit bench/BENCH_SHA256.txt to GitHub now, then run the baseline (Part 7).")

## 5. MODE = 'train': build training and validation files (Tue 6 Oct)

In [ ]:
if MODE == "train":
    bench = [json.loads(l) for l in open(fetch("bench/ilera_bench.jsonl"), encoding="utf-8")]
    recorded = dict(l.split()[::-1] for l in open(fetch("bench/BENCH_SHA256.txt")) if l.strip())
    assert sha256(fetch("bench/ilera_bench.jsonl")) == recorded["ilera_bench.jsonl"], \
        "bench/ilera_bench.jsonl does not match BENCH_SHA256.txt: the frozen benchmark was changed"
    bench_ids = {x["id"] for x in bench}
    bench_en_ids = {x["en_id"] for x in bench}
    print(f"frozen benchmark OK: {len(bench)} items")

    # candidates: English train rows, English bench-pool rows NOT used in the benchmark,
    # translated train rows, referral cases
    cand = ([r for r in en if r["split"] == "train" or (r["split"] == "bench" and r["id"] not in bench_ids)]
            + [r for r in hy if r["split"] == "train"] + ref)
    why = collections.Counter()
    kept = []
    for r in cand:
        if r["id"] in bench_ids:
            why["is a benchmark item"] += 1
        elif r["en_id"] in bench_en_ids:
            why["translation of a benchmark item"] += 1
        elif r.get("chunk_id") in probe_chunks:
            why["from a probe chunk"] += 1
        else:
            kept.append(r)
    print(f"{len(cand)} candidates -> {len(kept)} after leak rules", dict(why))

### Near-duplicate removal
A training question is removed if it is too similar to any benchmark question in the same language
(English training rows are compared with English benchmark questions, and so on). Exact duplicate
questions inside the training data are also removed.

In [ ]:
if MODE == "train":
    from rapidfuzz import fuzz, process
    bench_q = collections.defaultdict(list)
    for x in bench:
        bench_q[x["lang"]].append(x["question"])
    seen, dedup, near, exact = set(), [], [], 0
    for r in kept:
        key = (r["lang"], " ".join(r["question"].lower().split()))
        if key in seen:
            exact += 1
            continue
        hit = process.extractOne(r["question"], bench_q.get(r["lang"], []), scorer=fuzz.token_set_ratio,
                                 score_cutoff=DEDUP_THRESHOLD)
        if hit:
            near.append((r["id"], round(hit[1]), hit[0][:70]))
            continue
        seen.add(key)
        dedup.append(r)
    print(f"removed {len(near)} near-duplicates of benchmark questions and {exact} exact duplicates "
          f"-> {len(dedup)} rows")
    for n in near[:5]:
        print("   e.g.", n)

### Chat format, length check, validation split
Every domain row becomes `system + user + assistant`. Rows longer than `MAX_TOKENS` (N-ATLaS
tokenizer) are dropped. Validation = 5% (at least 150) of the domain rows, stratified by language; it
is used to pick the best training run, never the benchmark.

In [ ]:
if MODE == "train":
    def to_msg(r):
        return {"messages": [{"role": "system", "content": SYSTEM},
                             {"role": "user", "content": r["question"].strip()},
                             {"role": "assistant", "content": r["answer"]}],
                "id": r["id"], "lang": r["lang"], "kind": r.get("kind") or "referral",
                "action": r["action"], "chunk_id": r.get("chunk_id", "")}
    rows = [to_msg(r) for r in dedup]

    try:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained("NCAIR1/N-ATLaS", token=HF_TOKEN)
        n_tokens = lambda m: len(tok.apply_chat_template(m["messages"], tokenize=True))
        print("length check with the N-ATLaS tokenizer")
    except Exception as e:
        n_tokens = lambda m: int(2.0 * sum(len(x["content"].split()) for x in m["messages"]))
        print("tokenizer unavailable, using a word-count estimate:", str(e)[:80])
    too_long = [m["id"] for m in rows if n_tokens(m) > MAX_TOKENS]
    rows = [m for m in rows if m["id"] not in set(too_long)]
    print(f"dropped {len(too_long)} rows longer than {MAX_TOKENS} tokens")

    rng = random.Random(3407)
    by_lang = collections.defaultdict(list)
    for m in rows:
        by_lang[m["lang"]].append(m)
    n_val = max(VAL_MIN, int(VAL_FRACTION * len(rows)))
    val, train = [], []
    for lang, ms in by_lang.items():
        rng.shuffle(ms)
        k = round(n_val * len(ms) / len(rows))
        val += ms[:k]; train += ms[k:]
    print(f"domain rows: train {len(train)} | val {len(val)}")

### General-language data (Aya, Apache-2.0)
About 10% extra rows of ordinary human-written instructions in Hausa, Yoruba, Igbo and English,
**without** the Ilera system prompt. This keeps N-ATLaS's normal chat ability (avoids "catastrophic
forgetting") while the system prompt switches on the structured health-worker format.

In [ ]:
if MODE == "train":
    from datasets import load_dataset
    aya = load_dataset("CohereLabs/aya_dataset", split="train")
    aya = aya.filter(lambda x: x["language_code"] in ("hau", "yor", "ibo", "eng")
                     and 20 <= len(x["targets"]) <= 1500 and len(x["inputs"]) <= 1500)
    n_aya = int(AYA_FRACTION * len(train))
    per_lang = collections.defaultdict(list)
    for i, code_ in enumerate(aya["language_code"]):
        per_lang[code_].append(i)
    pick = []
    for code_, idx in sorted(per_lang.items()):             # balanced across the four languages
        rng.shuffle(idx)
        pick += idx[: n_aya // 4]
    LMAP = {"hau": "ha", "yor": "yo", "ibo": "ig", "eng": "en"}
    aya_rows = [{"messages": [{"role": "user", "content": aya[i]["inputs"]},
                              {"role": "assistant", "content": aya[i]["targets"]}],
                 "id": f"AYA-{i}", "lang": LMAP[aya[i]["language_code"]], "kind": "general",
                 "action": "", "chunk_id": ""} for i in pick]
    train += aya_rows
    rng.shuffle(train)
    print(f"added {len(aya_rows)} Aya rows", dict(collections.Counter(r["lang"] for r in aya_rows)))

### Write, check, upload

In [ ]:
if MODE == "train":
    write_jsonl("train.jsonl", train)
    write_jsonl("val.jsonl", val)
    assert not {m["id"] for m in train + val} & bench_ids, "benchmark item leaked into training"
    assert not {m["chunk_id"] for m in train + val} & probe_chunks, "probe chunk leaked into training"
    stats = {
        "train": len(train), "val": len(val),
        "train_by_lang": dict(collections.Counter(m["lang"] for m in train)),
        "train_by_kind": dict(collections.Counter(m["kind"] for m in train)),
        "train_action": dict(collections.Counter(m["action"] for m in train if m["action"])),
        "val_by_lang": dict(collections.Counter(m["lang"] for m in val)),
        "removed": {"leak_rules": dict(why), "near_duplicates": len(near), "exact_duplicates": exact,
                    "too_long": len(too_long)},
        "bench_sha256": recorded["ilera_bench.jsonl"],
    }
    json.dump(stats, open("train_stats.json", "w"), indent=2)
    print(json.dumps(stats, indent=1))
    domain = stats["train"] - stats["train_by_kind"].get("general", 0)
    if not 3000 <= domain + len(val) <= 5000:
        print(f"NOTE: {domain + len(val)} domain examples; the plan target is 3,000 to 5,000.")
    upload([("train.jsonl", "train.jsonl"), ("val.jsonl", "val.jsonl"), ("train_stats.json", "train_stats.json")],
           "03b_build: training files")

## Done when
* **Mon 5 Oct:** `bench/ilera_bench.jsonl` frozen (660 items: 300 en + 150 ha + 150 yo + 60 probe),
  `bench/afrimedqa_ids.txt` (300), and `BENCH_SHA256.txt` committed to GitHub. Then Part 7 (baseline).
* **Tue 6 Oct:** `train.jsonl` + `val.jsonl` on Hugging Face (3,000 to 5,000 domain examples + ~10% Aya),
  `train_stats.json` saved for the model card. Then Part 8 (training run A).